# 1. Imports, Loading Dataset

In [ ]:
import os
import joblib
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, confusion_matrix
)
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

DATA_DIR = Path('data')
SEED = 42
LABEL = 'isFraud'
PRIMARY_METRIC = 'Average Precision (PR-AUC)'

In [2]:
# Locate files
def locate(filename):
    candidates = [Path(filename), DATA_DIR / filename]
    if DATA_DIR.exists():
        candidates += sorted(DATA_DIR.glob(f"*-{filename}"))
    for path in candidates:
        if path.is_file():
            return path
    return None

paths = {name: locate(name) for name in ["train.csv", "test.csv", "val.csv"]}

# Load data
train = pd.read_csv(paths["train.csv"])
val = pd.read_csv(paths["val.csv"])
test = pd.read_csv(paths["test.csv"])

print(f"Train shape: {train.shape}, Validation shape: {val.shape}, Test shape: {test.shape}")

Train shape: (440018, 488), Validation shape: (61196, 488), Test shape: (89326, 488)


In [3]:
EXCLUDE = [LABEL, 'TransactionID', 'TransactionDT', 'TransactionDateTime', 'Transaction_Day', 'uid']
FEATURES = [c for c in train.columns if c not in EXCLUDE]

X_train, y_train = train.loc[:, FEATURES], train[LABEL].astype(int).to_numpy()
print("Initialised train features")
X_val, y_val = val.loc[:, FEATURES], val[LABEL].astype(int).to_numpy()
print("Initialised validation features")

Initialised train features
Initialised validation features


# 2. Preprocessing

In [4]:
# 1. Replace infinite values (+inf and -inf) with NaN across all datasets
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)

# instead of nan might want to change this to inf = max finite value and -inf = min finite value

# 2. Identify non-numeric columns
cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
print(f"Encoding {len(cat_cols)} categorical columns: {cat_cols}")

# 3. Encode categorical columns to numeric integers
if cat_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    
    # Fit and transform on training set, transform on validation set
    X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols].astype(str))
    X_val[cat_cols] = encoder.transform(X_val[cat_cols].astype(str))

Encoding 37 categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'P_email_company', 'device_company', 'card1_addr1', 'card4_card6', 'ProductCD_card6', 'P_email_company_device_company']


# 3. Baseline experiments

In [5]:
SAMPLING_RATIOS = [0.10, 0.25, 0.50, 1.00]
num_pos = np.sum(y_train == 1)
num_neg = np.sum(y_train == 0)
POS_WEIGHT = num_neg / num_pos
print(f"Calculated POS_WEIGHT: {POS_WEIGHT:.2f}")

Calculated POS_WEIGHT: 27.56


In [6]:
def make_models(weighted=False):
    return {
        "Logistic_Regression": Pipeline([
            ('imputer', SimpleImputer(strategy='median')), # replace missing with median
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                random_state=SEED,
                max_iter=1000,
                verbose=1,
                class_weight='balanced' if weighted else None
            )),
        ]),
        "LightGBM": LGBMClassifier(
            random_state=SEED,
            scale_pos_weight=POS_WEIGHT if weighted else 1.0,
            verbosity=-1,
            n_jobs=-1,
        ),
        'XGBoost': XGBClassifier(
            random_state=SEED,
            scale_pos_weight=POS_WEIGHT if weighted else 1.0,
            eval_metric='logloss',
            n_jobs=-1,
        ),
    }

In [7]:
results_list = []
best_candidates = {}

def run_experiment(name, condition, model, X_train, y_train):
    print(f'Running {name} / {condition} ...')

    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(X_train, y_train)
    convergence_warning = any(issubclass(w.category, ConvergenceWarning) for w in caught)

    for warning in caught:
        print(f'{name}/{condition}: {warning.message}')

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    metrics = {
        "Model": name, 
        "Condition": condition,
        "Recall": recall_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "F1_Score": f1_score(y_val, y_pred),
        "ROC AUC": roc_auc_score(y_val, y_prob),
        "Average Precision (PR-AUC)": average_precision_score(y_val, y_prob),
        "False Positive Rate": fpr,
        "True Positives": tp,
        "False Positives": fp,
        "False Negatives": fn,
        'Time taken': time.perf_counter() - started,
    }

    results_list.append(metrics)
    previous = best_candidates.get(name)
    if not convergence_warning and (previous is None or metrics[PRIMARY_METRIC] > previous['metrics'][PRIMARY_METRIC]):
        best_candidates[name] = {'model': model, 'metrics': metrics.copy()}
        print(f'  New best {name}: {PRIMARY_METRIC}={metrics[PRIMARY_METRIC]:.5f}')
    return metrics

def show_results():
    return pd.DataFrame(results_list).sort_values(
            [PRIMARY_METRIC, 'Model', 'Condition'], 
            ascending=[False, True, True]
        ).reset_index(drop=True)

In [8]:
# free system RAM (if needed)
import gc
gc.collect()

20

In [9]:
for name, model in make_models(weighted=False).items():
    run_experiment(name, 'Baseline', model, X_train, y_train)

baseline_results = show_results()
display(baseline_results)

Running Logistic_Regression / Baseline ...
  New best Logistic_Regression: Average Precision (PR-AUC)=0.58607
Running LightGBM / Baseline ...
  New best LightGBM: Average Precision (PR-AUC)=0.66667
Running XGBoost / Baseline ...
  New best XGBoost: Average Precision (PR-AUC)=0.68630


,Model,Condition,Recall,Precision,F1_Score,ROC AUC,Average Precision (PR-AUC),False Positive Rate,True Positives,False Positives,False Negatives,Time taken
0,XGBoost,Baseline,0.485541,0.842915,0.616159,0.944619,0.686303,0.003285,1041,194,1103,69.603471
1,LightGBM,Baseline,0.493937,0.829287,0.619117,0.947687,0.666673,0.003692,1059,218,1085,36.505654
2,Logistic_Regression,Baseline,0.413713,0.791258,0.543338,0.936062,0.586067,0.003963,887,234,1257,189.798901


## Interpretations
**1. Primary Metric Performance (PR-AUC)<br>**
XGBoost leads the baseline leaderboard: XGBoost achieves the highest PR-AUC score of 0.6863, driven by the highest Precision (84.29%) and lowest False Positive Rate (0.33%). Out of 1,235 positive predictions, it generated only 194 false alarms. LightGBM follows closely: LightGBM achieves a PR-AUC of 0.6667 and slightly outperforms XGBoost in $F_1$-Score (0.6191) and ROC-AUC (0.9477). It captures 1,059 true fraud instances (18 more than XGBoost) at the cost of slightly higher false positives (218 vs. 194).

**2. High Precision vs. Low Recall (The Default Threshold Problem)**
- Conservative Predictions at $p = 0.5$: All three baseline models exhibit high Precision (79.1% – 84.3%) but poor Recall (41.4% – 49.4%).
- High Cost of Uncaught Fraud: Because the default decision threshold is set to $0.5$, models fail to flag over half of all actual fraud cases, resulting in 1,085 to 1,257 undetected fraudulent transactions across validation samples.
- Justification for Imbalance Interventions: These baseline results confirm that class imbalance severely suppresses minority class probability estimates, providing a clear baseline justification for applying cost-sensitive weighting, resampling, or threshold tuning.

**3. Compute Efficiency & Runtime Analysis<br>**
LightGBM is the most efficientIts histogram-based binning makes it the most viable model for fast experimentation.Logistic Regression is a computational bottleneck as it suffers from slow gradient convergence due to unscaled features across a high-dimensional space (480+ columns).

# 3. Imbalance Experiments

| Sampler / Technique | Logistic Regression | LightGBM | XGBoost | Key Trade-offs & Practical Impact |
| :--- | :--- | :--- | :--- | :--- |
| **Weighted Loss (No Resampling)** | Supported natively via `class_weight='balanced'`[cite: 1]. Directly scales gradients for minority class samples. | Supported natively via `scale_pos_weight`[cite: 1]. Scales loss gradients without changing dataset size or speed. | Supported natively via `scale_pos_weight`[cite: 1]. Directly scales loss gradients for positive instances. | **Fastest & cleanest baseline.** Preserves 100% of data without adding synthetic noise or increasing memory usage. |
| **Random Under-Sampling (RUS)** | Significantly speeds up convergence. Helps move the linear hyperplane toward minority samples. | Speeds up training dramatically. May discard subtle non-fraud patterns needed by deep trees. | Accelerates tree-building speed. Risk of dropping rare non-fraud edge cases in high-dimensional space. | **Maximum training speedup.** Great for fast experimentation, but high risk of information loss from majority class. |
| **SMOTE (Oversampling)** | Creates synthetic points along feature vectors. Works reasonably well if numerical features are scaled. | Increases training set size and slows iterations. Trees often prefer direct loss weighting over synthetic interpolation. | Increases matrix size and slows histogram construction. Synthetic points can create non-existent fraud patterns. | **Preserves majority information**, but increases training time significantly and can introduce synthetic noise in sparse/categorical data. |
| **SMOTEENN / SMOTETomek (Hybrid)** | Cleans boundary overlap (via ENN/Tomek), producing a clearer separation linear decision boundary. | Cleans noisy boundary samples, but the preprocessing step ($k$-NN search) can become a bottleneck on large datasets. | Removes conflicting synthetic samples, but preprocessing computation often outweighs tree performance gains. | **Highest sample quality at decision boundaries**, but extremely slow to run on large tabular datasets (like IEEE-CIS Fraud). |

In [10]:
samplers = {
    "Weighted_Loss": None,  # Handled via scale_pos_weight / class_weight
    "Random_UnderSampling": RandomUnderSampler(sampling_strategy=0.1, random_state=SEED),
    "SMOTE": SMOTE(sampling_strategy=0.1, random_state=SEED),
}

In [11]:
for sampler_name, sampler in samplers.items():
    if sampler_name == "Random_UnderSampling":
        print(f"\n--- Resampling training data with {sampler_name} ---")
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        is_weighted = False
    elif sampler is not None:
        print(f"\n--- Resampling training data with {sampler_name} ---")
        imputer = SimpleImputer(strategy='median')
        X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
        X_res, y_res = sampler.fit_resample(X_train_imp, y_train)
        is_weighted = False
    else:
        X_res, y_res = X_train, y_train
        is_weighted = True
        
    models = make_models(weighted=is_weighted)
    
    for model_name, model in models.items():
        condition_label = f"{model_name}_{sampler_name}"
        run_experiment(model_name, condition_label, model, X_res, y_res)

Running Logistic_Regression / Logistic_Regression_Weighted_Loss ...
Running LightGBM / LightGBM_Weighted_Loss ...
  New best LightGBM: Average Precision (PR-AUC)=0.67490
Running XGBoost / XGBoost_Weighted_Loss ...

--- Resampling training data with Random_UnderSampling ---
Running Logistic_Regression / Logistic_Regression_Random_UnderSampling ...
Running LightGBM / LightGBM_Random_UnderSampling ...
  New best LightGBM: Average Precision (PR-AUC)=0.69490
Running XGBoost / XGBoost_Random_UnderSampling ...
  New best XGBoost: Average Precision (PR-AUC)=0.70105

--- Resampling training data with SMOTE ---
Running Logistic_Regression / Logistic_Regression_SMOTE ...
  New best Logistic_Regression: Average Precision (PR-AUC)=0.59329
Running LightGBM / LightGBM_SMOTE ...
Running XGBoost / XGBoost_SMOTE ...


In [12]:
imbalance_results = show_results()
display(imbalance_results)

,Model,Condition,Recall,Precision,F1_Score,ROC AUC,Average Precision (PR-AUC),False Positive Rate,True Positives,False Positives,False Negatives,Time taken
0,XGBoost,XGBoost_Random_UnderSampling,0.590951,0.751929,0.661792,0.947089,0.701045,0.007079,1267,418,877,27.776190
1,LightGBM,LightGBM_Random_UnderSampling,0.569030,0.755418,0.649109,0.949699,0.694901,0.006689,1220,395,924,25.121493
2,XGBoost,Baseline,0.485541,0.842915,0.616159,0.944619,0.686303,0.003285,1041,194,1103,69.603471
3,XGBoost,XGBoost_Weighted_Loss,0.769590,0.356680,0.487445,0.941417,0.685424,0.050396,1650,2976,494,59.617972
4,LightGBM,LightGBM_Weighted_Loss,0.821362,0.276626,0.413866,0.946757,0.674896,0.077982,1761,4605,383,49.882817
5,LightGBM,Baseline,0.493937,0.829287,0.619117,0.947687,0.666673,0.003692,1059,218,1085,36.505654
6,LightGBM,LightGBM_SMOTE,0.537313,0.678845,0.599844,0.925383,0.625510,0.009229,1152,545,992,65.171379
7,Logistic_Regression,Logistic_Regression_SMOTE,0.523321,0.673469,0.588976,0.936329,0.593286,0.009212,1122,544,1022,299.168727
8,Logistic_Regression,Baseline,0.413713,0.791258,0.543338,0.936062,0.586067,0.003963,887,234,1257,189.798901
9,Logistic_Regression,Logistic_Regression_Random_UnderSampling,0.512127,0.657485,0.575773,0.937963,0.582711,0.009686,1098,572,1046,76.046293


`XGBoost_Random_UnderSampling` achieves the highest overall PR-AUC of 0.7010, marking a notable improvement over the baseline (0.6863). It balances high Recall (59.10%) with strong Precision (75.19%), yielding the highest overall $F_1$-score (0.6618) while maintaining a minimal False Positive Rate (0.0071).

## Imbalance Strategy Interpretations
Random Undersampling is the optimal strategy
- Boosted Minority Signal: RUS successfully moved decision boundaries closer to minority fraud clusters without distorting feature geometry. Compared to baseline XGBoost, RUS increased True Positives from 1,041 to 1,267 (+21.7%) at the cost of modest additional false alarms (418 vs. 194).
- Computation Efficiency: By discarding majority non-fraud samples during training, RUS reduced fit times by ~50–60% compared to baseline models (~25s vs. ~50–70s).

# Save trained model

In [ ]:
# MODEL_DIR = "saved_models"
# os.makedirs(MODEL_DIR, exist_ok=True)

# sampler = RandomUnderSampler(sampling_strategy=0.1, random_state=SEED) # can change

# if sampler is not None:
#     X_res, y_res = sampler.fit_resample(X_train, y_train)
#     is_weighted = False
# else:
#     X_res, y_res = X_train, y_train
#     is_weighted = True

# model_name = "XGBoost" # can change
# model = make_models(weighted=False)[model_name]

# condition_label = f"{model_name}_RUS"
# run_experiment(model_name, condition_label, model, X_res, y_res)

# joblib.dump(model, MODEL_DIR / "xgboost_rus.joblib") # can change
# print("Model saved to xgboost_rus.joblib") # can change

Running XGBoost / XGBoost_RUS ...
  New best XGBoost: Average Precision (PR-AUC)=0.70105
Model saved to xgboost_rus.joblib


In [14]:
# # to reload model
# loaded_model = joblib.load("lightgbm_smote.joblib")